In [ ]:

import math

# ==========================================
#       MOVIE RECOMMENDATION SYSTEM
# ==========================================

movies = {
    1: {"name": "Inception", "genre": ["Sci-Fi", "Thriller"], "rating": 8.8},
    2: {"name": "Interstellar", "genre": ["Sci-Fi", "Drama"], "rating": 8.7},
    3: {"name": "The Dark Knight", "genre": ["Action", "Crime"], "rating": 9.0},
    4: {"name": "Titanic", "genre": ["Romance", "Drama"], "rating": 7.9},
    5: {"name": "Avengers Endgame", "genre": ["Action", "Sci-Fi"], "rating": 8.4},
    6: {"name": "The Notebook", "genre": ["Romance", "Drama"], "rating": 7.8},
    7: {"name": "Joker", "genre": ["Drama", "Thriller"], "rating": 8.4},
    8: {"name": "Avatar", "genre": ["Sci-Fi", "Adventure"], "rating": 7.9},
    9: {"name": "John Wick", "genre": ["Action", "Thriller"], "rating": 7.4},
    10: {"name": "Frozen", "genre": ["Animation", "Adventure"], "rating": 7.4},
    11: {"name": "Toy Story", "genre": ["Animation", "Comedy"], "rating": 8.3},
    12: {"name": "The Conjuring", "genre": ["Horror", "Thriller"], "rating": 7.5},
    13: {"name": "Coco", "genre": ["Animation", "Drama"], "rating": 8.4},
    14: {"name": "The Matrix", "genre": ["Sci-Fi", "Action"], "rating": 8.7},
    15: {"name": "La La Land", "genre": ["Romance", "Musical"], "rating": 8.0}
}

# Sample user ratings for collaborative filtering
user_ratings = {
    "Alice": {1: 5, 2: 5, 3: 4, 5: 5, 8: 4, 14: 5},
    "Bob": {3: 5, 4: 4, 5: 5, 6: 4, 9: 5, 15: 4},
    "Charlie": {1: 5, 2: 4, 7: 5, 8: 5, 12: 4, 14: 5},
    "David": {4: 5, 6: 5, 10: 4, 11: 5, 13: 5, 15: 5},
    "Emma": {3: 4, 5: 5, 7: 4, 9: 5, 14: 5}
}

# ==========================================
# DISPLAY ALL MOVIES
# ==========================================

def display_movies():
    print("\n========== AVAILABLE MOVIES ==========")

    for movie_id, movie in movies.items():
        print(
            f"{movie_id}. {movie['name']} | "
            f"Genre: {', '.join(movie['genre'])} | "
            f"Rating: {movie['rating']}/10"
        )

# ==========================================
# CONTENT-BASED FILTERING
# ==========================================

def content_recommendations(preferred_genres, watched):
    recommendations = []

    for movie_id, movie in movies.items():

        if movie_id in watched:
            continue

        matching_genres = set(preferred_genres) & set(movie["genre"])

        if matching_genres:
            score = len(matching_genres) / len(
                set(preferred_genres) | set(movie["genre"])
            )

            recommendations.append(
                (movie_id, score, movie["rating"])
            )

    recommendations.sort(
        key=lambda x: (x[1], x[2]),
        reverse=True
    )

    return recommendations


# ==========================================
# COLLABORATIVE FILTERING
# ==========================================

def cosine_similarity(user1, user2):

    common = set(user1) & set(user2)

    if not common:
        return 0

    dot_product = sum(
        user1[m] * user2[m] for m in common
    )

    magnitude1 = math.sqrt(
        sum(user1[m] ** 2 for m in common)
    )

    magnitude2 = math.sqrt(
        sum(user2[m] ** 2 for m in common)
    )

    if magnitude1 == 0 or magnitude2 == 0:
        return 0

    return dot_product / (magnitude1 * magnitude2)


def collaborative_recommendations(user_name, watched):

    if user_name not in user_ratings:
        return []

    current_user = user_ratings[user_name]
    scores = {}

    for other_user, ratings in user_ratings.items():

        if other_user == user_name:
            continue

        similarity = cosine_similarity(
            current_user, ratings
        )

        if similarity <= 0:
            continue

        for movie_id, rating in ratings.items():

            if movie_id not in watched:
                if movie_id not in scores:
                    scores[movie_id] = [0, 0]

                scores[movie_id][0] += similarity * rating
                scores[movie_id][1] += similarity

    recommendations = []

    for movie_id, values in scores.items():

        if values[1] > 0:
            predicted_rating = values[0] / values[1]

            recommendations.append(
                (movie_id, predicted_rating)
            )

    recommendations.sort(
        key=lambda x: x[1],
        reverse=True
    )

    return recommendations


# ==========================================
# COMBINED RECOMMENDATION SYSTEM
# ==========================================

def recommend_movies(user_name, preferred_genres, watched):

    content = content_recommendations(
        preferred_genres, watched
    )

    collaborative = collaborative_recommendations(
        user_name, watched
    )

    combined_scores = {}

    # Content-based score
    for movie_id, score, rating in content:
        combined_scores[movie_id] = (
            0.5 * score + 0.5 * (rating / 10)
        )

    # Collaborative score
    for movie_id, predicted_rating in collaborative:

        collaborative_score = predicted_rating / 5

        if movie_id in combined_scores:
            combined_scores[movie_id] = (
                0.5 * combined_scores[movie_id]
                + 0.5 * collaborative_score
            )
        else:
            combined_scores[movie_id] = collaborative_score

    results = sorted(
        combined_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    return results


# ==========================================
# TOP RATED MOVIES
# ==========================================

def top_rated_movies():

    print("\n========== TOP RATED MOVIES ==========")

    sorted_movies = sorted(
        movies.items(),
        key=lambda x: x[1]["rating"],
        reverse=True
    )

    for movie_id, movie in sorted_movies[:5]:

        print(
            f"{movie['name']} - "
            f"{movie['rating']}/10"
        )


# ==========================================
# MAIN PROGRAM
# ==========================================

def main():

    print("\n" + "=" * 45)
    print("       MOVIE RECOMMENDATION SYSTEM")
    print("=" * 45)

    print("\n1. View All Movies")
    print("2. Get Movie Recommendations")
    print("3. View Top Rated Movies")
    print("4. Exit")

    while True:

        choice = input("\nEnter your choice: ").strip()

        if choice == "1":

            display_movies()

        elif choice == "2":

            user_name = input(
                "\nEnter your name: "
            ).strip()

            print("\nAvailable genres:")
            genres = sorted({
                genre
                for movie in movies.values()
                for genre in movie["genre"]
            })

            print(", ".join(genres))

            preferred_genres = input(
                "\nEnter your preferred genres "
                "(comma separated): "
            )

            preferred_genres = [
                genre.strip().title()
                for genre in preferred_genres.split(",")
                if genre.strip()
            ]

            watched = set()

            print("\nEnter watched movie IDs separated by commas.")
            print("Leave blank if you haven't watched any.")

            watched_input = input(
                "Watched movie IDs: "
            ).strip()

            if watched_input:
                try:
                    watched = {
                        int(x.strip())
                        for x in watched_input.split(",")
                        if x.strip()
                    }

                    watched = watched & set(movies.keys())

                except ValueError:
                    print("Invalid IDs. Starting with an empty watched list.")
                    watched = set()

            # Save new user ratings
            if user_name not in user_ratings:
                user_ratings[user_name] = {}

                print("\nRate some movies to improve recommendations.")
                print("Enter ratings from 1 to 5, or press Enter to skip.")

                for movie_id, movie in movies.items():

                    if movie_id in watched:

                        rating = input(
                            f"Rate {movie['name']}: "
                        ).strip()

                        if rating:
                            try:
                                rating = int(rating)

                                if 1 <= rating <= 5:
                                    user_ratings[user_name][movie_id] = rating
                                else:
                                    print("Rating must be between 1 and 5.")

                            except ValueError:
                                print("Invalid rating. Skipped.")

            recommendations = recommend_movies(
                user_name,
                preferred_genres,
                watched
            )

            print("\n" + "=" * 45)
            print("       YOUR PERSONALIZED MOVIES")
            print("=" * 45)

            if recommendations:

                for movie_id, score in recommendations[:5]:

                    movie = movies[movie_id]

                    print(
                        f"\nMovie: {movie['name']}"
                        f"\nGenre: {', '.join(movie['genre'])}"
                        f"\nIMDb-style rating: {movie['rating']}/10"
                        f"\nRecommendation score: {score:.2f}"
                    )

            else:

                print("No recommendations found.")

        elif choice == "3":

            top_rated_movies()

        elif choice == "4":

            print("\nThank you for using Movie Recommendation System!")
            break

        else:

            print("Invalid choice. Please select 1, 2, 3, or 4.")


# ==========================================
# RUN PROJECT
# ==========================================

if __name__ == "__main__":
    main()



       MOVIE RECOMMENDATION SYSTEM

1. View All Movies
2. Get Movie Recommendations
3. View Top Rated Movies
4. Exit
